<a href="https://colab.research.google.com/github/rajasripalla1211-hub/Rajasri-/blob/main/F%20and%20B%20Artifacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================
# CELL 1: INSTALLATION
# ==============================

!pip -q install gradio plotly numpy

In [2]:
# ==============================
# CELL 2: BACKEND MATHEMATICS
# ==============================

import numpy as np


def rotation_matrix(angle):
    """
    Creates a 2D rotation matrix.
    """
    theta = np.radians(angle)

    return np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])


def rotate_point(angle, x, y):
    """
    Applies rotation matrix to a coordinate.
    """
    Q = rotation_matrix(angle)

    point = np.array([x, y])

    rotated_point = Q @ point

    return Q, rotated_point


def orthogonality_check(Q):
    """
    Checks Q^T Q = I.
    """
    QTQ = Q.T @ Q
    identity = np.eye(2)

    is_orthogonal = np.allclose(QTQ, identity)

    determinant = np.linalg.det(Q)

    return QTQ, determinant, is_orthogonal


def calculate_rotation(angle, x, y):

    Q, rotated_point = rotate_point(angle, x, y)

    QTQ, determinant, is_orthogonal = orthogonality_check(Q)

    return {
        "Q": Q,
        "rotated_point": rotated_point,
        "QTQ": QTQ,
        "determinant": determinant,
        "is_orthogonal": is_orthogonal
    }

In [3]:
# ==============================
# CELL 3: BACKEND GRAPH
# ==============================

import plotly.graph_objects as go


def create_rotation_graph(angle):

    # Original phone screen
    width = 4
    height = 7

    corners = np.array([
        [-width / 2, -height / 2],
        [ width / 2, -height / 2],
        [ width / 2,  height / 2],
        [-width / 2,  height / 2],
        [-width / 2, -height / 2]
    ])

    # Rotation matrix
    Q = rotation_matrix(angle)

    # Rotate screen
    rotated_corners = corners @ Q.T

    fig = go.Figure()

    # Original screen
    fig.add_trace(
        go.Scatter(
            x=corners[:, 0],
            y=corners[:, 1],
            mode="lines+markers",
            name="Original Portrait",
            line=dict(width=4)
        )
    )

    # Rotated screen
    fig.add_trace(
        go.Scatter(
            x=rotated_corners[:, 0],
            y=rotated_corners[:, 1],
            mode="lines+markers",
            name=f"Rotated {angle}°",
            line=dict(width=4)
        )
    )

    # Coordinate axes
    fig.add_hline(y=0)
    fig.add_vline(x=0)

    fig.update_layout(
        title=f"Screen Rotation: {angle}°",
        xaxis_title="X-axis",
        yaxis_title="Y-axis",
        xaxis=dict(scaleanchor="y"),
        yaxis=dict(scaleratio=1),
        height=550,
        template="plotly_white",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        )
    )

    return fig

In [4]:
# ==============================
# CELL 4: BACKEND PROCESSING
# ==============================

def process_rotation(angle, x, y):

    result = calculate_rotation(angle, x, y)

    Q = result["Q"]
    rotated = result["rotated_point"]
    QTQ = result["QTQ"]
    determinant = result["determinant"]
    is_orthogonal = result["is_orthogonal"]

    matrix_text = (
        "Rotation Matrix Q\n\n"
        f"┌ {Q[0,0]:8.4f}  {Q[0,1]:8.4f} ┐\n"
        f"└ {Q[1,0]:8.4f}  {Q[1,1]:8.4f} ┘"
    )

    coordinate_text = (
        "Coordinate Transformation\n\n"
        f"Original point  : ({x:.2f}, {y:.2f})\n"
        f"Rotated point   : "
        f"({rotated[0]:.2f}, {rotated[1]:.2f})"
    )

    orthogonal_text = (
        "Orthogonality Test\n\n"
        "QᵀQ =\n"
        f"[[{QTQ[0,0]:.4f}, {QTQ[0,1]:.4f}],\n"
        f" [{QTQ[1,0]:.4f}, {QTQ[1,1]:.4f}]]\n\n"
        f"det(Q) = {determinant:.4f}\n\n"
        f"{'✓ Q is Orthogonal' if is_orthogonal else '✗ Q is not Orthogonal'}"
    )

    graph = create_rotation_graph(angle)

    return (
        matrix_text,
        coordinate_text,
        orthogonal_text,
        graph
    )

In [5]:
# ==============================
# CELL 5: FRONTEND
# ==============================

import gradio as gr


with gr.Blocks(
    title="Screen Rotation as Orthogonal Transformation"
) as app:

    # --------------------------------
    # HEADER
    # --------------------------------

    gr.Markdown(
        """
        # 📱 Screen Rotation as Orthogonal Transformation

        ### B.Tech 1st Year Mathematics Mini Project

        **Orthogonal Matrices • Rotation Matrices • Coordinate Transformation**
        """
    )

    gr.Markdown("---")

    # --------------------------------
    # INPUT SECTION
    # --------------------------------

    gr.Markdown("## ⚙️ Rotation Controls")

    with gr.Row():

        with gr.Column():

            angle_input = gr.Slider(
                minimum=0,
                maximum=360,
                value=90,
                step=90,
                label="Rotation Angle"
            )

            x_input = gr.Number(
                value=2,
                label="X Coordinate"
            )

            y_input = gr.Number(
                value=3,
                label="Y Coordinate"
            )

            rotate_button = gr.Button(
                "🔄 Apply Rotation",
                variant="primary"
            )

        with gr.Column():

            gr.Markdown(
                """
                ### Rotation Modes

                **0°** → Original screen

                **90°** → Portrait → Landscape

                **180°** → Upside down

                **270°** → Reverse rotation

                **360°** → Original position
                """
            )

    gr.Markdown("---")

    # --------------------------------
    # GRAPH
    # --------------------------------

    gr.Markdown("## 📊 Interactive Screen Visualization")

    graph_output = gr.Plot(
        label="Screen Rotation"
    )

    # --------------------------------
    # RESULTS
    # --------------------------------

    gr.Markdown("---")

    gr.Markdown("## 🧮 Mathematical Results")

    with gr.Row():

        matrix_output = gr.Textbox(
            label="Rotation Matrix",
            lines=6
        )

        coordinate_output = gr.Textbox(
            label="Coordinate Transformation",
            lines=6
        )

        orthogonal_output = gr.Textbox(
            label="Orthogonality Check",
            lines=9
        )

    # --------------------------------
    # THEORY
    # --------------------------------

    gr.Markdown(
        """
        ---

        ## 📐 Mathematical Verification

        For a rotation matrix **Q**:

        **QᵀQ = I**

        Therefore, **Q is an orthogonal matrix**.

        Also:

        **det(Q) = +1**

        Rotation preserves **distances, lengths and angles**.
        """
    )

    # --------------------------------
    # 180 DEGREE DEMONSTRATION
    # --------------------------------

    gr.Markdown("---")

    gr.Markdown("## 🔁 Two 90° Rotations")

    twice_button = gr.Button(
        "Show 90° + 90° = 180°"
    )

    twice_output = gr.Textbox(
        label="Two Rotation Result",
        lines=8
    )

    def two_rotations():

        Q90 = rotation_matrix(90)

        Q180 = Q90 @ Q90

        return (
            "90° Rotation Matrix:\n\n"
            f"{np.round(Q90, 4)}\n\n"
            "Q × Q = Q²:\n\n"
            f"{np.round(Q180, 4)}\n\n"
            "✓ Therefore, two 90° rotations produce a 180° rotation."
        )

    # --------------------------------
    # BUTTON CONNECTIONS
    # --------------------------------

    rotate_button.click(
        fn=process_rotation,
        inputs=[
            angle_input,
            x_input,
            y_input
        ],
        outputs=[
            matrix_output,
            coordinate_output,
            orthogonal_output,
            graph_output
        ]
    )

    twice_button.click(
        fn=two_rotations,
        inputs=[],
        outputs=twice_output
    )


# --------------------------------
# LAUNCH
# --------------------------------

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa09e1414438d42540.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
